# Proposta Marcelo - Leitura de dados

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcelo7bastos/relatorio_dados_damei/blob/main/notebooks/proposta_marcelo.ipynb)

Este notebook inicia o fluxo do projeto. O objetivo desta etapa é configurar a origem dos dados, listar as bases disponíveis e confirmar que conseguimos ler os arquivos Excel necessários para o relatório estadual.

## 1. Importar bibliotecas

Usamos bibliotecas padrão do Python para caminhos, datas e ambiente, além de `pandas` para ler as planilhas Excel.

In [ ]:
from datetime import datetime
from pathlib import Path
import os

import pandas as pd


## 2. Configurar origem dos dados e pasta de saída

O GitHub guarda código, documentação e template. O Google Drive é o repositório operacional dos dados e dos relatórios gerados.

Existem apenas dois modos:

- `local`: roda no VS Code e usa uma cópia local/mock em `dados_brutos/dado_atual`.
- `google_drive`: roda no Google Colab, monta o Google Drive e lê os dados oficiais do Drive.

In [ ]:
# O modo é escolhido automaticamente:
# - Google Colab: "google_drive"
# - VS Code/local: "local"
MODO_DADOS = "google_drive" if "COLAB_RELEASE_TAG" in os.environ else "local"

# UF usada nas próximas etapas do projeto.
UF_INTERESSE = "MG"

# Caminhos no Google Drive para execução no Colab.
GOOGLE_DRIVE_RAW_DIR = Path("/content/drive/MyDrive/MDA/DAMEI_Relatorio_Dados/dados_brutos/dado_atual")
GOOGLE_DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/MDA/dado_atual")

# Subpastas usadas somente no modo local.
RAW_SUBDIR = Path("dados_brutos") / "dado_atual"
OUTPUT_SUBDIR = Path("relatorios_gerados")

# Data de execução usada futuramente no nome dos arquivos gerados.
RUN_TIMESTAMP = datetime.now()
RUN_YYYYMM = RUN_TIMESTAMP.strftime("%Y%m")
RUN_DATETIME = RUN_TIMESTAMP.strftime("%Y%m%d%H%M%S")


## 3. Resolver caminhos

Esta célula transforma a configuração acima em caminhos finais de leitura e gravação.

No modo `local`, o notebook usa as pastas do próprio repositório. No modo `google_drive`, o notebook exige Google Colab, monta o Drive e usa os caminhos `/content/drive/...`.

In [ ]:
def esta_no_colab() -> bool:
    """Retorna True quando o notebook está rodando no Google Colab."""
    return "COLAB_RELEASE_TAG" in os.environ


def encontrar_raiz_projeto(inicio: Path | None = None) -> Path:
    """Procura a raiz do projeto apenas no modo local."""
    inicio = (inicio or Path.cwd()).resolve()

    for candidato in [inicio, *inicio.parents]:
        tem_notebooks = (candidato / "notebooks").exists()
        tem_requirements = (candidato / "requirements.txt").exists()
        if tem_notebooks and tem_requirements:
            return candidato

    raise FileNotFoundError(
        "Não encontrei a raiz do projeto para o modo local. "
        "Abra o VS Code na pasta relatorio_dados_damei ou execute o notebook "
        "a partir de uma subpasta do repositório."
    )


if MODO_DADOS == "local":
    PROJECT_ROOT = encontrar_raiz_projeto()
    DATA_PROJECT_ROOT = PROJECT_ROOT
    RAW_DIR = PROJECT_ROOT / RAW_SUBDIR
    OUTPUT_BASE_DIR = PROJECT_ROOT / OUTPUT_SUBDIR
    OUTPUT_DIR = OUTPUT_BASE_DIR / RUN_YYYYMM
elif MODO_DADOS == "google_drive":
    if not esta_no_colab():
        raise RuntimeError(
            'MODO_DADOS = "google_drive" só deve ser usado no Google Colab. '
            'No VS Code local, use MODO_DADOS = "local".'
        )

    from google.colab import drive

    drive.mount("/content/drive")
    DATA_PROJECT_ROOT = GOOGLE_DRIVE_RAW_DIR.parents[1]
    RAW_DIR = GOOGLE_DRIVE_RAW_DIR
    OUTPUT_DIR = GOOGLE_DRIVE_OUTPUT_DIR
else:
    raise ValueError('MODO_DADOS deve ser "local" ou "google_drive".')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Modo de dados:     {MODO_DADOS}")
print(f"UF de interesse:   {UF_INTERESSE}")
print(f"Projeto de dados:  {DATA_PROJECT_ROOT}")
print(f"Pasta de dados:    {RAW_DIR}")
print(f"Pasta de saída:    {OUTPUT_DIR}")


## 4. Validar pastas de dados e saída

Antes de ler qualquer arquivo, validamos se a pasta de dados existe. A pasta de saída é criada automaticamente quando ainda não existir. No modo `google_drive`, ela usa o caminho configurado em `GOOGLE_DRIVE_OUTPUT_DIR`.

In [ ]:
if not RAW_DIR.exists():
    raise FileNotFoundError(f"A pasta de dados não existe: {RAW_DIR}")

print("Pasta de dados encontrada.")
print("Pasta de saída pronta.")


## 5. Listar arquivos Excel

Aqui verificamos quais arquivos `.xlsx` estão disponíveis na pasta de dados configurada.

In [ ]:
excel_files = sorted(RAW_DIR.glob("*.xlsx"))

if not excel_files:
    raise FileNotFoundError(f"Nenhum arquivo .xlsx encontrado em {RAW_DIR}")

print(f"Foram encontrados {len(excel_files)} arquivos Excel:")
for arquivo in excel_files:
    tamanho_mb = arquivo.stat().st_size / (1024 * 1024)
    print(f"- {arquivo.name} ({tamanho_mb:.2f} MB)")


## 6. Identificar arquivos por política

Nesta etapa fazemos um primeiro mapeamento automático entre nomes de arquivos e políticas públicas. Essa lógica ainda é simples, mas já ajuda a validar se temos todas as bases esperadas.

In [ ]:
arquivos_encontrados = {
    "caf_dap": None,
    "ater": None,
    "pronaf": None,
    "mais_alimentos": None,
    "pncf": None,
    "garantia_safra": None,
    "pnra": None,
}

for arquivo in excel_files:
    nome = arquivo.name.lower()

    if "caf" in nome:
        arquivos_encontrados["caf_dap"] = arquivo
    elif "ater" in nome:
        arquivos_encontrados["ater"] = arquivo
    elif "pronaf" in nome:
        arquivos_encontrados["pronaf"] = arquivo
    elif "mais_alimentos" in nome or "mais-alimentos" in nome:
        arquivos_encontrados["mais_alimentos"] = arquivo
    elif "pncf" in nome:
        arquivos_encontrados["pncf"] = arquivo
    elif "garantia-safra" in nome or "garantia_safra" in nome:
        arquivos_encontrados["garantia_safra"] = arquivo
    elif "pnra" in nome:
        arquivos_encontrados["pnra"] = arquivo

resumo_arquivos = pd.DataFrame(
    [
        {
            "politica": politica,
            "arquivo": arquivo.name if arquivo else "NÃO ENCONTRADO",
            "encontrado": arquivo is not None,
        }
        for politica, arquivo in arquivos_encontrados.items()
    ]
)

resumo_arquivos


## 7. Inspecionar abas e colunas

Agora abrimos cada arquivo com `pandas.ExcelFile`, listamos as abas e coletamos os nomes das colunas. Este teste ainda não transforma os dados; ele apenas confirma que os arquivos são legíveis.

In [ ]:
linhas = []

for politica, arquivo in arquivos_encontrados.items():
    if arquivo is None:
        linhas.append(
            {
                "politica": politica,
                "arquivo": "NÃO ENCONTRADO",
                "aba": None,
                "qtd_colunas": None,
                "colunas": None,
            }
        )
        continue

    xls = pd.ExcelFile(arquivo)
    for aba in xls.sheet_names:
        colunas = pd.read_excel(arquivo, sheet_name=aba, nrows=0).columns.astype(str).tolist()
        linhas.append(
            {
                "politica": politica,
                "arquivo": arquivo.name,
                "aba": aba,
                "qtd_colunas": len(colunas),
                "colunas": ", ".join(colunas[:12]) + (" ..." if len(colunas) > 12 else ""),
            }
        )

df_estrutura = pd.DataFrame(linhas)
df_estrutura


## 8. Testar leitura das abas principais

Por fim, carregamos as abas principais de cada política e mostramos o tamanho de cada tabela. Se esta célula rodar sem erro, o teste de leitura está aprovado.

In [ ]:
abas_principais = {
    "caf_dap": "DADOS",
    "ater": "DADOS",
    "pronaf": "Dados",
    "mais_alimentos": "Dados",
    "pncf": "DADOS",
    "garantia_safra": "DADOS",
    "pnra": "DADOS",
}

tabelas = {}
resumo_leitura = []

for politica, aba in abas_principais.items():
    arquivo = arquivos_encontrados.get(politica)

    if arquivo is None:
        resumo_leitura.append(
            {
                "politica": politica,
                "status": "arquivo não encontrado",
                "linhas": None,
                "colunas": None,
            }
        )
        continue

    df = pd.read_excel(arquivo, sheet_name=aba)
    tabelas[politica] = df
    resumo_leitura.append(
        {
            "politica": politica,
            "status": "ok",
            "linhas": df.shape[0],
            "colunas": df.shape[1],
        }
    )

pd.DataFrame(resumo_leitura)


## 9. Incorporação da proposta Wesley

A partir daqui incorporamos os avanços do notebook `proposta_wesley.ipynb`: uso dos totalizadores de CAF e PRONAF, comparação UF x Brasil e geração de um primeiro relatório Word de teste. A implementação abaixo usa os caminhos já configurados neste notebook, sem caminhos fixos e sem download automático.

## 10. Funções auxiliares

Estas funções padronizam validação, percentuais e formatação brasileira de números antes da geração do relatório.

In [ ]:
try:
    from docx import Document
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])
    from docx import Document
    from docx.enum.text import WD_ALIGN_PARAGRAPH


def normalizar_uf(uf: str) -> str:
    return str(uf).strip().upper()


def validar_colunas(df: pd.DataFrame, colunas: list[str], contexto: str) -> None:
    faltantes = [col for col in colunas if col not in df.columns]
    if faltantes:
        raise KeyError(f"Colunas ausentes em {contexto}: {faltantes}")


def calcular_percentual(parte: float, total: float) -> float:
    if pd.isna(total) or total == 0:
        return 0.0
    return float(parte) / float(total) * 100


def formatar_inteiro(valor: float) -> str:
    return f"{valor:,.0f}".replace(",", ".")


def formatar_moeda(valor: float) -> str:
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def formatar_percentual(valor: float) -> str:
    return f"{valor:.2f}%".replace(".", ",")


## 11. Carregar totalizadores de CAF e PRONAF

O notebook do Wesley avançou usando as abas de totalizadores. Isso é útil para o relatório estadual, porque essas abas já trazem os dados consolidados por UF.

In [ ]:
arquivo_caf = arquivos_encontrados.get("caf_dap")
arquivo_pronaf = arquivos_encontrados.get("pronaf")

if arquivo_caf is None:
    raise FileNotFoundError("Arquivo CAF/DAP não encontrado.")
if arquivo_pronaf is None:
    raise FileNotFoundError("Arquivo PRONAF não encontrado.")

df_caf_totalizadores = pd.read_excel(
    arquivo_caf,
    sheet_name="TOTALIZADORES",
    engine="openpyxl",
    na_values=["NA", "na", ""],
    skiprows=6,
    nrows=27,
)
df_caf_totalizadores.columns = df_caf_totalizadores.columns.str.strip()

caf_colunas_texto = ["dt_referencia", "dt_geracao", "cod_ibge_estados", "uf"]
caf_colunas_numericas = [
    "Soma de CAFs PF ATIVO",
    "Soma de CAFs PJ ATIVO",
    "Soma de QUANTIDADE DE MULHERES EM CAF ATIVO",
    "Soma de QUANTIDADE DE HOMENS EM CAF ATIVO",
]
validar_colunas(df_caf_totalizadores, caf_colunas_texto + caf_colunas_numericas, "CAF totalizadores")

for col in caf_colunas_texto:
    df_caf_totalizadores[col] = df_caf_totalizadores[col].astype(str)
for col in caf_colunas_numericas:
    df_caf_totalizadores[col] = pd.to_numeric(df_caf_totalizadores[col], errors="coerce").fillna(0)

df_pronaf_totalizadores = pd.read_excel(
    arquivo_pronaf,
    sheet_name="Totalizadores",
    engine="openpyxl",
    na_values=["NA", "na", ""],
)
df_pronaf_totalizadores.columns = df_pronaf_totalizadores.columns.str.strip()

pronaf_colunas_texto = ["dt_referencia", "dt_geracao", "uf", "cod_ibge_uf", "ANO"]
pronaf_colunas_numericas = [
    "qtd_contratos_1_Feminino",
    "qtd_contratos_1_Masculino",
    "qtd_contratos_1_Sem_Identificacao",
    "valor_total_contratos_1_Feminino",
    "valor_total_contratos_1_Masculino",
    "valor_total_contratos_1_Sem_Identificacao",
    "qtd_operacoes_1_Feminino",
    "qtd_operacoes_1_Masculino",
    "qtd_operacoes_1_Sem_Identificacao",
    "ticket_medio_1_Feminino",
    "ticket_medio_1_Masculino",
    "ticket_medio_1_Sem_Identificacao",
]
validar_colunas(df_pronaf_totalizadores, pronaf_colunas_texto + pronaf_colunas_numericas, "PRONAF totalizadores")

for col in pronaf_colunas_texto:
    df_pronaf_totalizadores[col] = df_pronaf_totalizadores[col].astype(str)
for col in pronaf_colunas_numericas:
    df_pronaf_totalizadores[col] = pd.to_numeric(df_pronaf_totalizadores[col], errors="coerce").fillna(0)

pd.DataFrame(
    [
        {"base": "CAF totalizadores", "linhas": df_caf_totalizadores.shape[0], "colunas": df_caf_totalizadores.shape[1]},
        {"base": "PRONAF totalizadores", "linhas": df_pronaf_totalizadores.shape[0], "colunas": df_pronaf_totalizadores.shape[1]},
    ]
)


## 12. Calcular indicadores UF x Brasil

Aqui consolidamos os indicadores de CAF e PRONAF para a UF escolhida e calculamos a participação da UF em relação ao Brasil.

In [ ]:
UF_INTERESSE = normalizar_uf(UF_INTERESSE)

df_caf_uf = df_caf_totalizadores[df_caf_totalizadores["uf"].str.upper() == UF_INTERESSE].copy()
if df_caf_uf.empty:
    raise ValueError(f"UF {UF_INTERESSE} não encontrada nos totalizadores de CAF.")

caf_linhas = []
for indicador, coluna in [
    ("CAFs Pessoa Física", "Soma de CAFs PF ATIVO"),
    ("CAFs Pessoa Jurídica", "Soma de CAFs PJ ATIVO"),
    ("Mulheres em CAF ativo", "Soma de QUANTIDADE DE MULHERES EM CAF ATIVO"),
    ("Homens em CAF ativo", "Soma de QUANTIDADE DE HOMENS EM CAF ATIVO"),
]:
    valor_uf = df_caf_uf[coluna].sum()
    valor_br = df_caf_totalizadores[coluna].sum()
    caf_linhas.append(
        {
            "indicador": indicador,
            "uf": valor_uf,
            "brasil": valor_br,
            "percentual_uf_br": calcular_percentual(valor_uf, valor_br),
        }
    )
df_caf_resumo = pd.DataFrame(caf_linhas)

df_pronaf_uf = df_pronaf_totalizadores[df_pronaf_totalizadores["uf"].str.upper() == UF_INTERESSE].copy()
if df_pronaf_uf.empty:
    raise ValueError(f"UF {UF_INTERESSE} não encontrada nos totalizadores de PRONAF.")

pronaf_linhas = []
for publico, sufixo in [
    ("Feminino", "Feminino"),
    ("Masculino", "Masculino"),
    ("Sem identificação", "Sem_Identificacao"),
]:
    coluna_qtd = f"qtd_contratos_1_{sufixo}"
    coluna_valor = f"valor_total_contratos_1_{sufixo}"
    qtd_uf = df_pronaf_uf[coluna_qtd].sum()
    qtd_br = df_pronaf_totalizadores[coluna_qtd].sum()
    valor_uf = df_pronaf_uf[coluna_valor].sum()
    valor_br = df_pronaf_totalizadores[coluna_valor].sum()
    pronaf_linhas.append(
        {
            "publico": publico,
            "qtd_uf": qtd_uf,
            "qtd_brasil": qtd_br,
            "percentual_qtd": calcular_percentual(qtd_uf, qtd_br),
            "valor_uf": valor_uf,
            "valor_brasil": valor_br,
            "percentual_valor": calcular_percentual(valor_uf, valor_br),
        }
    )

df_pronaf_resumo = pd.DataFrame(pronaf_linhas)
df_pronaf_resumo.loc[len(df_pronaf_resumo)] = {
    "publico": "Total",
    "qtd_uf": df_pronaf_resumo["qtd_uf"].sum(),
    "qtd_brasil": df_pronaf_resumo["qtd_brasil"].sum(),
    "percentual_qtd": calcular_percentual(df_pronaf_resumo["qtd_uf"].sum(), df_pronaf_resumo["qtd_brasil"].sum()),
    "valor_uf": df_pronaf_resumo["valor_uf"].sum(),
    "valor_brasil": df_pronaf_resumo["valor_brasil"].sum(),
    "percentual_valor": calcular_percentual(df_pronaf_resumo["valor_uf"].sum(), df_pronaf_resumo["valor_brasil"].sum()),
}

df_caf_resumo, df_pronaf_resumo


## 13. Gerar relatório Word de teste

Esta é uma primeira geração `.docx` baseada na proposta do Wesley. O arquivo é salvo em `OUTPUT_DIR`, que no Colab aponta para o Google Drive e no modo local aponta para `relatorios_gerados/AAAAMM`.

In [ ]:
def adicionar_cabecalho_tabela(tabela, cabecalhos: list[str]) -> None:
    for idx, titulo in enumerate(cabecalhos):
        cell = tabela.rows[0].cells[idx]
        cell.text = titulo
        for paragraph in cell.paragraphs:
            for run in paragraph.runs:
                run.bold = True


doc = Document()

titulo = doc.add_heading(f"Relatório de Dados Teste: {UF_INTERESSE}", level=0)
titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_paragraph(f"Gerado em: {RUN_TIMESTAMP.strftime('%d/%m/%Y %H:%M')}")
doc.add_paragraph("Documento preliminar gerado a partir dos totalizadores de CAF e PRONAF.")

doc.add_heading(f"1. Situação do CAF ({UF_INTERESSE} x Brasil)", level=1)
doc.add_paragraph("Indicadores consolidados de CAF ativo por UF, comparados ao total nacional.")

table_caf = doc.add_table(rows=1, cols=4)
table_caf.style = "Table Grid"
adicionar_cabecalho_tabela(table_caf, ["Indicador", UF_INTERESSE, "Brasil", "% UF/BR"])

for _, row in df_caf_resumo.iterrows():
    cells = table_caf.add_row().cells
    cells[0].text = str(row["indicador"])
    cells[1].text = formatar_inteiro(row["uf"])
    cells[2].text = formatar_inteiro(row["brasil"])
    cells[3].text = formatar_percentual(row["percentual_uf_br"])

doc.add_heading(f"2. Desempenho do PRONAF ({UF_INTERESSE} x Brasil)", level=1)
doc.add_paragraph("Indicadores de contratos e valores do PRONAF por público, comparados ao total nacional.")

table_pronaf = doc.add_table(rows=1, cols=7)
table_pronaf.style = "Table Grid"
adicionar_cabecalho_tabela(
    table_pronaf,
    ["Público", f"Qtd. {UF_INTERESSE}", "Qtd. Brasil", "% Qtd", f"Valor {UF_INTERESSE}", "Valor Brasil", "% Valor"],
)

for _, row in df_pronaf_resumo.iterrows():
    cells = table_pronaf.add_row().cells
    cells[0].text = str(row["publico"])
    cells[1].text = formatar_inteiro(row["qtd_uf"])
    cells[2].text = formatar_inteiro(row["qtd_brasil"])
    cells[3].text = formatar_percentual(row["percentual_qtd"])
    cells[4].text = formatar_moeda(row["valor_uf"])
    cells[5].text = formatar_moeda(row["valor_brasil"])
    cells[6].text = formatar_percentual(row["percentual_valor"])
    if row["publico"] == "Total":
        for cell in cells:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.bold = True

nome_relatorio = f"relatorio_teste_{UF_INTERESSE.lower()}_caf_pronaf_{RUN_DATETIME}.docx"
caminho_relatorio = OUTPUT_DIR / nome_relatorio
doc.save(caminho_relatorio)

print(f"Relatório Word gerado em: {caminho_relatorio}")
